# 04 Gold - Banking

**Audience:** participants learning the AIDP medallion pattern with PySpark.

**Prerequisites:** use the lab's shared compute and run the previous notebook first.

**Learning goal:** Builds the industry KPIs from accepted Silver records.

## Outline

1. Inspect the participant-scoped inputs.
2. Transform and persist this medallion layer.
3. Register external tables when this layer owns them.
4. Verify the row counts printed by the final statements.


In [ ]:
import re
# oidlUtils is injected by AIDP Workbench; no import is required.

def required_parameter(name):
    value = oidlUtils.parameters.getParameter(name, "")
    if value is None or not str(value).strip():
        raise ValueError(f"Missing AIDP job parameter: {name}")
    return str(value).strip()

participant_key = required_parameter("participant_key")
lab_id = required_parameter("lab_id")
workspace_root = required_parameter("workspace_root")
bucket_name = required_parameter("bucket_name")
objectstorage_namespace = required_parameter("objectstorage_namespace")
catalog_name = required_parameter("catalog_name")

participant_match = re.fullmatch(r"u([1-9][0-9]*)", participant_key)
if participant_match is None or int(participant_match.group(1)) < 101:
    raise ValueError("Invalid participant_key")
if lab_id != 'banking':
    raise ValueError("This notebook belongs to a different lab")
if not workspace_root.startswith("/Workspace/medallon/"):
    raise ValueError("Invalid workspace_root")
if catalog_name != f"{participant_key}_aidp_lab":
    raise ValueError("Invalid participant catalog")
spark.conf.set("spark.aidp.lineage.enabled", "true")

def table(layer, logical_name):
    prefix = f"{lab_id}_"
    physical_name = logical_name if logical_name.startswith(prefix) else prefix + logical_name
    return f"{catalog_name}.oci_{layer}.{participant_key}_{physical_name}"

from pyspark.sql import functions as F

silver = {"accounts": f"oci://{bucket_name}@{objectstorage_namespace}/03_silver/users/{participant_key}/banking/accounts/", "branches": f"oci://{bucket_name}@{objectstorage_namespace}/03_silver/users/{participant_key}/banking/branches/", "customers": f"oci://{bucket_name}@{objectstorage_namespace}/03_silver/users/{participant_key}/banking/customers/", "transactions": f"oci://{bucket_name}@{objectstorage_namespace}/03_silver/users/{participant_key}/banking/transactions/"}
gold = {"branch_daily": f"oci://{bucket_name}@{objectstorage_namespace}/04_gold/users/{participant_key}/banking/banking_branch_daily/", "customer_value": f"oci://{bucket_name}@{objectstorage_namespace}/04_gold/users/{participant_key}/banking/banking_customer_value/"}
customers = spark.table(table("silver", "customers"))
accounts = spark.table(table("silver", "accounts"))
transactions = spark.table(table("silver", "transactions"))
branches = spark.table(table("silver", "branches"))
latest_event = transactions.agg(F.max("event_time").alias("max_event_time"))
transactions_30d = (transactions.crossJoin(latest_event)
    .filter(F.col("event_time") >= F.col("max_event_time") - F.expr("INTERVAL 30 DAYS"))
    .drop("max_event_time"))
account_metrics = (accounts.groupBy("participant_key", "customer_id")
    .agg(F.countDistinct("account_id").alias("account_count"), F.sum("balance").alias("current_balance")))
transaction_metrics = (transactions_30d.join(
    accounts.select("participant_key", "account_id", "customer_id", "branch_id"),
    ["participant_key", "account_id"],
)
    .groupBy("participant_key", "customer_id")
    .agg(F.count("transaction_id").alias("transaction_count_30d"),
         F.sum(F.when(F.col("transaction_type") == "debit", F.col("amount")).otherwise(0)).alias("debit_amount_30d"),
         F.sum(F.when(F.col("transaction_type").isin("credit", "refund"), F.col("amount")).otherwise(0)).alias("credit_amount_30d"),
         F.max("event_time").alias("last_transaction_at")))
customer_value = customers.select("participant_key", "customer_id").join(account_metrics, ["participant_key", "customer_id"], "left").join(transaction_metrics, ["participant_key", "customer_id"], "left")
customer_value.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(table("gold", "customer_value"))
branch_daily = (transactions.join(
    accounts.select("participant_key", "account_id", "branch_id"),
    ["participant_key", "account_id"],
)
    .withColumn("business_date", F.to_date("event_time"))
    .groupBy("participant_key", "business_date", "branch_id")
    .agg(F.countDistinct("account_id").alias("active_accounts"), F.count("transaction_id").alias("transaction_count"), F.sum("amount").alias("transaction_amount"), F.avg("amount").alias("average_transaction_amount")))
branch_daily.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(table("gold", "branch_daily"))
customer_value.show(20, truncate=False)

for table_name in gold:
    row_count = spark.table(table("gold", table_name)).count()
    assert row_count > 0, f"Gold table {table_name} is empty"
    print(f"Gold {table_name}: {row_count} rows")


## Expected result

Two non-empty, industry-specific aggregate Delta tables are registered.

**Exercise:** rerun this notebook and confirm that counts do not increase. All writes use
participant-exclusive paths and overwrite mode, so a second run is idempotent.

**Common pitfall:** do not replace the participant paths with shared locations. That would mix
different students' data. As an extension, query the registered tables with `spark.sql`.
